# Pentagons in Triangles
Unit pentagons packed inside an equilateral triangle (s (side length), minimized). Run all cells top to bottom.

Per n: random restarts, warm start from n-1 (one more shape inserted), basin hopping, then a 60-digit validity
certificate (proves no overlap and full containment -- not proof of optimality), then SVG + JSON saved.
Resumes if interrupted in the same runtime (finished n are skipped). Extend `NS` below to widen the range.

**Increasing accuracy:** raise `SEARCH_SEC` / `HOP_SEC` in the loop cell if the printed `s` is still dropping
noticeably near the end of a phase; if it has flattened, more time won't help -- a different seed will.

In [ ]:
!pip -q install jax jaxlib scipy mpmath numpy

In [ ]:
import json
# Generic shape-in-shape packing engine: search, basin-hop, certify, SVG.
# An item is a list of convex "parts" (each an Mx2 local-vertex array); non-convex items
# (e.g. an L) are given as >1 convex part whose union is the shape. A container is either
# a regular N-gon (half-planes) or a circle (single radius bound).
import json, time
import numpy as np
import jax, jax.numpy as jnp
from scipy.optimize import minimize
import mpmath as mp

jax.config.update("jax_enable_x64", True)
mp.mp.dps = 60

# ---- set per notebook -------------------------------------------------
ITEM_PARTS = []          # list of np.ndarray (k,2): item's convex decomposition, local frame
CONTAINER = ("poly", 4)  # ("poly", N) for regular N-gon, or ("circle", None)
CTHETA = -np.pi / 2       # polygon container rotation: one edge horizontal, at the bottom
# -------------------------------------------------------------------------


def set_shape(parts, container):
    global ITEM_PARTS, CONTAINER
    ITEM_PARTS = [np.asarray(p, dtype=float) for p in parts]
    CONTAINER = container


def _rot_verts(x, y, th, local):
    """local (k,2) static -> (n,k,2) rotated+translated instances."""
    L = jnp.asarray(local)
    c, s = jnp.cos(th), jnp.sin(th)
    lx, ly = L[:, 0], L[:, 1]
    rx = c[:, None] * lx[None, :] - s[:, None] * ly[None, :]
    ry = s[:, None] * lx[None, :] + c[:, None] * ly[None, :]
    return jnp.stack([rx + x[:, None], ry + y[:, None]], axis=-1)


def unpack(z, n): return z[:n], z[n:2*n], z[2*n:3*n], z[3*n]


def _cnorm(n_c):
    return jnp.array([CTHETA + k * 2*np.pi/n_c for k in range(n_c)])


def cont_viol(z, n):
    x, y, th, s = unpack(z, n)
    kind, N = CONTAINER
    all_v = jnp.concatenate([_rot_verts(x, y, th, p) for p in ITEM_PARTS], axis=1)   # n,K,2
    if kind == "poly":
        apo = s / (2*jnp.tan(np.pi/N))
        nrm = jnp.stack([jnp.cos(_cnorm(N)), jnp.sin(_cnorm(N))], axis=1)             # N,2
        proj = jnp.einsum('nkc,mc->nkm', all_v, nrm)                                   # n,K,N
        return jnp.max(proj, axis=1) - apo                                             # n,N
    else:  # circle, s == radius
        d = jnp.linalg.norm(all_v, axis=-1)                                            # n,K
        return jnp.max(d, axis=-1, keepdims=True) - s                                  # n,1


def _convex_sat(Pa, Pb):
    """Pa,Pb: (...,ka,2),(...,kb,2). Separation via every edge normal of both. +ve = disjoint."""
    def edge_axes(P):
        nxt = jnp.roll(P, -1, axis=-2)
        e = nxt - P
        ax = jnp.stack([-e[..., 1], e[..., 0]], axis=-1)
        return ax / (jnp.linalg.norm(ax, axis=-1, keepdims=True) + 1e-15)
    A = jnp.concatenate([edge_axes(Pa), edge_axes(Pb)], axis=-2)          # ...,ka+kb,2
    pa = jnp.einsum('...vc,...ac->...av', Pa, A)
    pb = jnp.einsum('...vc,...ac->...av', Pb, A)
    sep = jnp.maximum(pb.min(-1) - pa.max(-1), pa.min(-1) - pb.max(-1))
    return jnp.max(sep, axis=-1)


def pair_sep(z, n):
    x, y, th, s = unpack(z, n)
    parts = [_rot_verts(x, y, th, p) for p in ITEM_PARTS]     # each n,k,2
    worst = None
    for a in range(len(parts)):
        for b in range(len(parts)):
            Pa = parts[a][:, None, :, :] * jnp.ones((1, n, 1, 1))
            Pb = parts[b][None, :, :, :] * jnp.ones((n, 1, 1, 1))
            sep = _convex_sat(Pa, Pb)
            worst = sep if worst is None else jnp.minimum(worst, sep)
    return worst


def energy(z, n):
    cv = jnp.maximum(cont_viol(z, n), 0.0)
    ms = pair_sep(z, n)
    iu = jnp.triu(jnp.ones((n, n)), k=1)
    pen = jnp.maximum(-ms, 0.0) * iu
    return jnp.sum(cv**2) + jnp.sum(pen**2)


def make_fns(n):
    def obj(z, mu): return z[3*n] + mu*energy(z, n)
    vg = jax.jit(jax.value_and_grad(obj))
    CV = jax.jit(lambda z: cont_viol(z, n))
    PS = jax.jit(lambda z: pair_sep(z, n))
    return vg, CV, PS


def max_violation(z, n, CV, PS):
    cv = float(jnp.max(CV(z)))
    if n < 2: return cv
    ms = np.array(PS(z)); iu = np.triu_indices(n, 1)
    return max(cv, float(-ms[iu].min()))


def penalty_solve(z0, n, vg, mus=(1e1,1e2,1e3,1e4,1e5,1e6), maxiter=400):
    z = z0.copy()
    for mu in mus:
        f = lambda zz: tuple(np.asarray(a, dtype=np.float64) for a in vg(zz, mu))
        r = minimize(f, z, jac=True, method="L-BFGS-B", options={"maxiter": maxiter, "maxcor": 30})
        z = r.x
    return z


def feasible_scale(z, n, CV, PS):
    x, y, th, s = unpack(jnp.asarray(z), n)
    def viol(k):
        zz = jnp.concatenate([x*k, y*k, th, jnp.array([s*k])])
        return max_violation(zz, n, CV, PS)
    if viol(1.0) <= 0: return np.array(z)
    hi = 1.0 + 1e-6
    while viol(hi) > 0:
        hi = 1.0 + (hi-1.0)*2
        if hi > 3: return None
    lo = 1.0
    for _ in range(40):
        mid = 0.5*(lo+hi)
        if viol(mid) > 0: lo = mid
        else: hi = mid
    k = hi
    return np.array(jnp.concatenate([x*k, y*k, th, jnp.array([s*k])]))


def slsqp_polish(z, n, CV, PS, iters=200):
    z = np.array(z)
    x, y = z[:n], z[n:2*n]
    D = np.hypot(x[:,None]-x[None,:], y[:,None]-y[None,:])
    reach = 2*max(np.linalg.norm(p, axis=1).max() for p in ITEM_PARTS) + 0.8
    I, J = np.where(np.triu(D < reach, 1))
    I = jnp.array(I); J = jnp.array(J)
    def cons(zz):
        zz = jnp.asarray(zz)
        cv = -cont_viol(zz, n).ravel()
        p = pair_sep(zz, n)[I, J]
        return jnp.concatenate([cv, p])
    cj = jax.jit(cons); cjac = jax.jit(jax.jacfwd(cons))
    res = minimize(lambda zz: zz[3*n], z, jac=lambda zz: np.eye(1, 3*n+1, 3*n)[0],
                   constraints=[{"type":"ineq","fun": lambda zz: np.asarray(cj(zz)),
                                 "jac": lambda zz: np.asarray(cjac(zz))}],
                   method="SLSQP", options={"maxiter": iters, "ftol": 1e-13})
    return res.x


def _item_area():
    # shoelace, summed over convex parts (parts are non-overlapping by construction)
    tot = 0.0
    for p in ITEM_PARTS:
        x, y = p[:,0], p[:,1]
        tot += 0.5*abs(np.dot(x, np.roll(y,-1)) - np.dot(y, np.roll(x,-1)))
    return tot


def _cont_area(s):
    kind, N = CONTAINER
    if kind == "poly": return (N/4)/np.tan(np.pi/N) * s*s
    return np.pi * s*s


def _cont_radius(s):
    kind, N = CONTAINER
    return s if kind == "circle" else s/(2*np.sin(np.pi/N))


def s_estimate(n, density=0.78):
    return np.sqrt(n*_item_area() / (density * _cont_area(1.0)))


def random_init(n, s0, rng):
    Rc = _cont_radius(s0)
    pts = []
    while len(pts) < n:
        ang = rng.random()*2*np.pi; rad = np.sqrt(rng.random())*Rc*0.85
        pts.append((rad*np.cos(ang), rad*np.sin(ang)))
    pts = np.array(pts); th = rng.random(n)*2*np.pi
    return np.concatenate([pts[:,0], pts[:,1], th, [s0]])


def run(n, seconds, seed=0, verbose=True, out=None):
    rng = np.random.default_rng(seed)
    vg, CV, PS = make_fns(n)
    best = None; t0=time.time(); trial=0
    while time.time()-t0 < seconds:
        trial += 1
        s_est = s_estimate(n)*(1.2+0.25*rng.random())
        z0 = random_init(n, s_est, rng)
        z = penalty_solve(z0, n, vg)
        zf = feasible_scale(z, n, CV, PS)
        if zf is None: continue
        if best is not None and zf[3*n] > best[3*n] + 0.3: continue
        try:
            zp = slsqp_polish(zf, n, CV, PS)
            zp2 = feasible_scale(zp, n, CV, PS)
            if zp2 is not None and zp2[3*n] < zf[3*n]: zf = zp2
        except Exception: pass
        s = zf[3*n]
        if best is None or s < best[3*n]:
            best = zf
            if verbose: print(f"n={n} trial={trial} t={time.time()-t0:.0f}s s={s:.6f}", flush=True)
            if out: json.dump({"n": n, "s": float(s), "z": best.tolist()}, open(out, "w"))
    return best


def hop(n, seconds, seed, start_z, out=None):
    rng = np.random.default_rng(seed)
    vg, CV, PS = make_fns(n)
    best = np.array(start_z); cur = best.copy(); t0=time.time(); it=0
    while time.time()-t0 < seconds:
        it += 1
        zz = cur.copy()
        k = rng.integers(1, max(2, n//4+1))
        idx = rng.choice(n, k, replace=False)
        sig = rng.choice([0.3,0.7,1.2])
        zz[idx] += rng.normal(0,sig,k); zz[n+idx] += rng.normal(0,sig,k)
        zz[2*n+idx] += rng.normal(0,0.5,k)
        if rng.integers(3)==0:
            j = rng.integers(n); Rc=_cont_radius(zz[3*n])
            ang=rng.random()*2*np.pi; rad=np.sqrt(rng.random())*Rc*0.85
            zz[j]=rad*np.cos(ang); zz[n+j]=rad*np.sin(ang); zz[2*n+j]=rng.random()*2*np.pi
        zz[3*n] = cur[3*n]*1.04
        z1 = penalty_solve(zz, n, vg, mus=(1e2,1e3,1e4,1e5,1e6), maxiter=300)
        zf = feasible_scale(z1, n, CV, PS)
        if zf is None or zf[3*n] > best[3*n]+0.05: continue
        try:
            zp = slsqp_polish(zf, n, CV, PS)
            z2 = feasible_scale(zp, n, CV, PS)
            if z2 is not None and z2[3*n] < zf[3*n]: zf = z2
        except Exception: pass
        s = zf[3*n]
        if s < cur[3*n]+0.01: cur = zf
        if s < best[3*n]-1e-9:
            best = zf; print(f"n={n} hop it={it} t={time.time()-t0:.0f}s s={s:.6f}", flush=True)
            if out: json.dump({"n": n, "s": float(s), "z": best.tolist()}, open(out, "w"))
    return best


def grow(zprev, n, rng, tries=6):
    m = n-1
    vg, CV, PS = make_fns(n); best=None
    x,y,th,s = zprev[:m], zprev[m:2*m], zprev[2*m:3*m], zprev[3*m]
    Rc=_cont_radius(s)
    for _ in range(tries):
        ang=rng.random()*2*np.pi; rad=np.sqrt(rng.random())*Rc*0.85
        z0 = np.concatenate([x,[rad*np.cos(ang)], y,[rad*np.sin(ang)], th,[rng.random()*2*np.pi],[s*1.06]])
        zf = feasible_scale(penalty_solve(z0,n,vg,mus=(1e2,1e3,1e4,1e5,1e6),maxiter=300), n, CV, PS)
        if zf is not None and (best is None or zf[3*n]<best[3*n]): best=zf
    return best


def certify(z, n, eps=1e-9):
    """60-digit proof: every item vertex strictly inside container; every part-pair strictly separated (SAT)."""
    z = [mp.mpf(float(v)) for v in z]; k = 1+mp.mpf(eps)
    x=[v*k for v in z[:n]]; y=[v*k for v in z[n:2*n]]; th=z[2*n:3*n]; s=z[3*n]*k
    kind, N = CONTAINER
    def rot(local, i):
        c,sN = mp.cos(th[i]), mp.sin(th[i])
        return [(c*lx - sN*ly + x[i], sN*lx + c*ly + y[i]) for lx,ly in local]
    parts = [[rot(p.tolist(), i) for i in range(n)] for p in ITEM_PARTS]
    cont = mp.inf
    if kind == "poly":
        apo = s/(2*mp.tan(mp.pi/N))
        cnorm = [CTHETA + kk*2*mp.pi/N for kk in range(N)]
        for i in range(n):
            for phi in cnorm:
                nx,ny = mp.cos(phi), mp.sin(phi)
                proj = max(vx*nx+vy*ny for part in parts for vx,vy in part[i])
                cont = min(cont, apo - proj)
    else:
        for i in range(n):
            dmax = max(mp.sqrt(vx*vx+vy*vy) for part in parts for vx,vy in part[i])
            cont = min(cont, s - dmax)
    def gap(A,B):
        best=-mp.inf
        for P in (A,B):
            for j in range(len(P)):
                ex,ey = P[(j+1)%len(P)][0]-P[j][0], P[(j+1)%len(P)][1]-P[j][1]
                nx,ny = ey,-ex; nn=mp.sqrt(nx*nx+ny*ny)
                pa=[(px*nx+py*ny)/nn for px,py in A]; pb=[(px*nx+py*ny)/nn for px,py in B]
                best=max(best, min(pb)-max(pa), min(pa)-max(pb))
        return best
    reach = mp.mpf(2*max(np.linalg.norm(p, axis=1).max() for p in ITEM_PARTS) + 1e-6)
    pair = mp.inf
    for i in range(n):
        for j in range(i+1, n):
            if mp.hypot(x[i]-x[j], y[i]-y[j]) > reach: continue
            worst = mp.inf
            for a in range(len(ITEM_PARTS)):
                for b in range(len(ITEM_PARTS)):
                    g = gap(parts[a][i], parts[b][j])
                    worst = min(worst, g)
            pair = min(pair, worst)
    assert cont > 0 and (n < 2 or pair > 0), (float(cont), float(pair))
    return s, cont, pair


def page_value(s): return f"{mp.floor(s*10**5)/10**5}+"


def geometry(z, n, W=250):
    s=float(z[3*n]); x,y,th = z[:n], z[n:2*n], z[2*n:3*n]
    kind, N = CONTAINER
    Rc = _cont_radius(s)
    pad=1; sc=(W-2*pad)/(2*Rc); H=2*Rc*sc+2*pad
    cx,cy = W/2, H/2
    P = lambda px,py: (cx+px*sc, cy-py*sc)
    items = []
    for p in ITEM_PARTS:
        Vn = np.array(_rot_verts(jnp.asarray(x), jnp.asarray(y), jnp.asarray(th), p))
        items.append([[P(vx,vy) for vx,vy in Vn[i]] for i in range(n)])
    if kind == "poly":
        cont = [P(Rc*np.cos(CTHETA+np.pi/N+k*2*np.pi/N), Rc*np.sin(CTHETA+np.pi/N+k*2*np.pi/N)) for k in range(N)]
    else:
        cont = [P(Rc*np.cos(a), Rc*np.sin(a)) for a in np.linspace(0, 2*np.pi, 96)]
    return items, cont, W, H


def write_svg(z, n, path):
    items, cont, W, H = geometry(z, n)
    pts = lambda v: " ".join(f"{a:.3f},{b:.3f}" for a,b in v)
    body = "".join(f'<polygon points="{pts(part[i])}" fill="#ffffe7" stroke="#000" stroke-width="0.75"/>'
                   for part in items for i in range(n))
    open(path,"w").write(f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H:.3f}" viewBox="0 0 {W} {H:.3f}">'
                          f'{body}<polygon points="{pts(cont)}" fill="none" stroke="#000" stroke-width="1.5"/></svg>')


import numpy as np

def regular(m):
    R = 1.0/(2*np.sin(np.pi/m))
    return np.array([(R*np.cos(k*2*np.pi/m), R*np.sin(k*2*np.pi/m)) for k in range(m)])

def domino():
    v = np.array([(0,0),(2,0),(2,1),(0,1)], dtype=float)
    return v - v.mean(axis=0)

def tan():
    v = np.array([(0,0),(1,0),(0,1)], dtype=float)
    return v - v.mean(axis=0)

def l_tromino():
    v = np.array([(0,0),(2,0),(2,1),(1,1),(1,2),(0,2)], dtype=float)
    c = v.mean(axis=0)
    a = np.array([(0,0),(2,0),(2,1),(0,1)]) - c
    b = np.array([(0,1),(1,1),(1,2),(0,2)]) - c
    return [a, b]

set_shape([regular(5)], ("poly", 3))


In [ ]:
NS = range(41, 51)
SEARCH_SEC = 240   # random restarts per n; raise if s is still dropping near the end of this phase
HOP_SEC = 120       # basin hopping per n; raise for higher n (see README: Increasing Accuracy)
OUT = "/content/pentagons_in_triangles"; import os; os.makedirs(OUT, exist_ok=True)
KNOWN = {}

prev, rows = None, []
for n in NS:
    jf = f"{OUT}/{n}.json"
    if os.path.exists(jf):
        d = json.load(open(jf)); prev = __import__("numpy").array(d["z"]); rows.append((n, d["s_cert"])); continue
    rng = __import__("numpy").random.default_rng(n)
    cands = [c for c in (run(n, SEARCH_SEC, n, verbose=False), grow(prev, n, rng) if prev is not None else None) if c is not None]
    if not cands:
        print(f"n={n}  no feasible layout found in budget -- skipping (rerun this cell later to retry)", flush=True)
        continue
    start = min(cands, key=lambda c: c[3*n])
    best = hop(n, HOP_SEC, n + 1000, start, "/tmp/hop.json")
    s_cert, cs, pg = certify(best, n)
    write_svg(best, n, f"{OUT}/pentagons_in_triangles_n{n}.svg")
    json.dump({"n": n, "s_cert": float(s_cert), "z": [float(v) for v in best]}, open(jf, "w"))
    rows.append((n, float(s_cert)))
    prev = best
    note = f"  page {KNOWN[n]}+ -> {'BEATS' if s_cert < KNOWN[n] else 'worse'}" if n in KNOWN else ""
    print(f"n={n}  certified s={mp.nstr(s_cert, 10)}  submit {page_value(s_cert)}{note}", flush=True)

import csv
with open(f"{OUT}/summary.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["n", "s_certified", "s_page_format"])
    for n, s in rows: w.writerow([n, s, page_value(mp.mpf(s))])
for (a, sa), (b, sb) in zip(rows, rows[1:]):
    if sb < sa: print(f"warning: s({b}) < s({a}); n={a} search is weak, rerun it longer")
import shutil
shutil.make_archive(OUT, "zip", OUT)
try:
    from google.colab import files; files.download(OUT + ".zip")
except ImportError: pass